# Day 12 – Used Car Data Preprocessing

## Objective
Perform a complete data preprocessing workflow on the Used Car Resale Dataset, including outlier handling, categorical encoding, feature scaling, train/test splitting, and leakage prevention.

## 1. Import Libraries and Load Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Day12_Used_Car_Preprocessing_Dataset.csv")
display(df.head())
print("Shape:", df.shape)

## 2. Initial Inspection

In [ ]:
print("Columns:", df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("Data Type"))
print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))
print("\nDuplicate rows:", df.duplicated().sum())
display(df.describe().T)

## 3. Separate Features and Target

`Resale_Price_Lakh` is the target variable. `Car_ID` is kept for record tracking but excluded from model features because it is only an identifier.

In [ ]:
target = "Resale_Price_Lakh"
id_col = "Car_ID"

X = df.drop(columns=[target])
y = df[target]

print("Features:", X.columns.tolist())
print("Target:", target)

## 4. Split into Training and Testing Sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 5. Decide the Encoding Strategy

**Nominal variables:** `Brand`, `Fuel_Type`, `Transmission`, `City`, and `Seller_Type` have no natural order, so One-Hot Encoding is used.

**Ordinal variable:** `Condition` has an order, so it is mapped as:
`Poor = 1`, `Fair = 2`, `Good = 3`, `Excellent = 4`.

**Numerical variables:** will be imputed if necessary, processed for outliers using IQR, and standardized.

In [ ]:
nominal_cols = ["Brand", "Fuel_Type", "Transmission", "City", "Seller_Type"]
ordinal_col = "Condition"
numeric_cols = [
    "Year", "Mileage_Km", "Engine_CC", "Power_BHP",
    "Previous_Owners", "Accidents_Reported", "Service_Score"
]
model_numeric_cols = numeric_cols + [ordinal_col]

condition_map = {
    "Poor": 1,
    "Fair": 2,
    "Good": 3,
    "Excellent": 4
}

X_train_model = X_train.drop(columns=[id_col]).copy()
X_test_model = X_test.drop(columns=[id_col]).copy()

X_train_model[ordinal_col] = X_train_model[ordinal_col].map(condition_map)
X_test_model[ordinal_col] = X_test_model[ordinal_col].map(condition_map)

display(X_train_model[[ordinal_col]].head())

## 6. Detect Outliers Using the IQR Method

IQR = Q3 − Q1.

Potential outlier limits are:
- Lower = Q1 − 1.5 × IQR
- Upper = Q3 + 1.5 × IQR

The limits are calculated **only from the training data** to prevent data leakage.

In [ ]:
Q1 = X_train_model[model_numeric_cols].quantile(0.25)
Q3 = X_train_model[model_numeric_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

display(pd.DataFrame({
    "Q1": Q1, "Q3": Q3, "IQR": IQR,
    "Lower_Bound": lower_bound,
    "Upper_Bound": upper_bound
}).round(2))

## 7. Create the Leakage-Safe Preprocessing Pipeline

The numerical pipeline performs:
1. Median imputation
2. IQR outlier clipping
3. Standardization using `StandardScaler`

The categorical pipeline performs:
1. Most-frequent imputation
2. One-Hot Encoding

The preprocessor is **fitted only on `X_train_model`** and then used to transform both train and test data.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

class IQRClipper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.columns_ = X_df.columns.tolist()
        q1 = X_df.quantile(0.25)
        q3 = X_df.quantile(0.75)
        iqr = q3 - q1
        self.lower_ = q1 - self.factor * iqr
        self.upper_ = q3 + self.factor * iqr
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.columns_)
        return X_df.clip(self.lower_, self.upper_, axis="columns").to_numpy()

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.columns_, dtype=object)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier_clipper", IQRClipper(factor=1.5)),
    ("scaler", StandardScaler())
])

nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, model_numeric_cols),
    ("nominal", nominal_pipeline, nominal_cols)
])

# Fit ONLY on training data
preprocessor.fit(X_train_model)

X_train_array = preprocessor.transform(X_train_model)
X_test_array = preprocessor.transform(X_test_model)

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_array, columns=feature_names, index=X_train_model.index
)
X_test_processed = pd.DataFrame(
    X_test_array, columns=feature_names, index=X_test_model.index
)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

## 8. Verify the Processed Data

In [ ]:
print("Missing values:", X_train_processed.isnull().sum().sum())
print("Infinite values:", np.isinf(X_train_processed.to_numpy()).sum())

print("\nProcessed training data:")
display(X_train_processed.head())

scaled_numeric = X_train_processed[
    [c for c in feature_names if c.startswith("num__")]
]
display(pd.DataFrame({
    "Mean": scaled_numeric.mean(),
    "Std": scaled_numeric.std(ddof=0)
}).round(3))

## 9. Export Preprocessed Train and Test Datasets

`Car_ID` and `Resale_Price_Lakh` are included in the exported files for record identification and target tracking. They were not used as model input features.

In [ ]:
train_export = pd.concat([
    X_train["Car_ID"].reset_index(drop=True),
    X_train_processed.reset_index(drop=True),
    y_train.reset_index(drop=True).rename("Resale_Price_Lakh")
], axis=1)

test_export = pd.concat([
    X_test["Car_ID"].reset_index(drop=True),
    X_test_processed.reset_index(drop=True),
    y_test.reset_index(drop=True).rename("Resale_Price_Lakh")
], axis=1)

train_export.to_csv("Day12_Used_Car_Preprocessed_Train.csv", index=False)
test_export.to_csv("Day12_Used_Car_Preprocessed_Test.csv", index=False)

print("Training file saved.")
print("Testing file saved.")
display(train_export.head())

## 10. Preprocessing Decisions Summary

| Task | Method | Reason |
|---|---|---|
| Identifier | Remove `Car_ID` from model inputs | Identifier is not a useful car characteristic |
| Missing numerical values | Median imputation | Robust and simple |
| Missing categorical values | Most-frequent imputation | Suitable for categorical data |
| Outliers | IQR clipping | Limits extreme values without deleting rows |
| Nominal variables | One-Hot Encoding | Categories have no natural order |
| Ordinal variable | Manual ordinal mapping | `Condition` has a meaningful order |
| Scaling | StandardScaler | Standardizes numerical features |
| Leakage prevention | Fit preprocessing on training set only | Prevents test-set information from influencing preprocessing |

## Conclusion

The dataset has been separated into features and target, split into training and testing sets, encoded, outlier-treated, and scaled. The transformations are fitted only on the training data, making the workflow suitable for subsequent machine-learning modeling without test-set leakage.